# 1.3 · GROUP BY 与聚合 / GROUP BY & Aggregation

> **课程定位 / Where this fits**
> **Part 1 第 3 课**。1.1 学会单表查询，1.2 学会多表 JOIN，这一节进入"**分组聚合宇宙**"——`GROUP BY`、`HAVING`、`ROLLUP`、`CUBE`、`GROUPING SETS`、`FILTER`。
> **Part 1, lesson 3.** Now into the multi-group aggregation universe.

> 📐 **符号约定 / Notation**
> 我们用 $g_i$ 表示分组 $i$，$n_i$ 表示该组的行数。
> $g_i$ = group $i$, $n_i$ = its row count.

> 💡 **面试相关 / Interview-relevant**
> - "WHERE 和 HAVING 的区别" ★★★★★（必考）
> - "为什么 `SELECT` 里非聚合列必须在 `GROUP BY` 里" ★★★★★
> - "用 SQL 算每个用户的转化漏斗" ★★★★（条件聚合）
> - "`ROLLUP` / `CUBE` 区别" ★★★（高级）
> - "`SELECT DISTINCT` vs `GROUP BY`" ★★★★（很多人答不上）
>
> Top hits: WHERE vs HAVING, GROUP BY constraint, conditional aggregation for funnels, ROLLUP vs CUBE, DISTINCT vs GROUP BY.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 解释 `GROUP BY` 的执行逻辑：先**分组**，再对每组施加聚合。
   Explain GROUP BY: split into groups, apply aggregate per group.
2. 用 `HAVING` 过滤"聚合后"的组——并说清和 `WHERE` 的差别。
   Filter post-aggregation with `HAVING` and contrast with `WHERE`.
3. 写出**条件聚合**（`SUM(CASE WHEN ...)` 或 `COUNT(... ) FILTER (WHERE ...)`）。
   Write conditional aggregates (the SUM(CASE WHEN) pattern, plus the modern FILTER clause).
4. 用 `ROLLUP` / `CUBE` / `GROUPING SETS` 一次性算出**小计 + 总计 + 多维度交叉**。
   Use ROLLUP / CUBE / GROUPING SETS for subtotals + grand totals + multi-dimensional crosstabs.
5. 知道 8 个 DS 常用的**进阶聚合**：`STDDEV`, `VAR`, `MEDIAN`, `PERCENTILE_CONT`, `MODE`, `STRING_AGG`, `ARRAY_AGG`, `BOOL_OR`/`BOOL_AND`。
   Know 8 DS-relevant advanced aggregates.
6. 调试 **GROUP BY 行数错乱** 的常见原因。
   Debug common GROUP BY bugs.

---

## 目录 / Table of Contents

1. [GROUP BY 基础 / The Basics](#1)
2. [WHERE vs HAVING ⭐](#2)
3. [`SELECT` 里能放什么——GROUP BY 的"对齐规则"](#3)
4. [DISTINCT vs GROUP BY](#4)
5. [多列分组 / Multi-column GROUP BY](#5)
6. [条件聚合 / Conditional Aggregation ⭐](#6)
7. [进阶聚合函数 / Advanced Aggregates](#7)
8. [`ROLLUP` —— 小计 + 总计 / Subtotals + Grand total](#8)
9. [`CUBE` —— 所有维度组合 / All-dim crosstab](#9)
10. [`GROUPING SETS` —— 自由组合](#10)
11. [⚠ NULL 在 GROUP BY 中的行为](#11)
12. [实战：销售分析报表 / Hands-on](#12)
13. [小结 / Summary](#13)


<a id="1"></a>
## 1. GROUP BY 基础 / The Basics

```sql
SELECT <grouping_cols>, <aggregates>
FROM   <table>
GROUP BY <grouping_cols>;
```

**执行逻辑** / Execution model:
1. **分组** Split：按 `GROUP BY` 列把行分桶
2. **聚合** Apply：对每桶里的所有行算 `COUNT/SUM/AVG/...`
3. **合并** Combine：每组返回**一行**

```
原始表             分组（按 genre）         聚合（COUNT, AVG）
┌──────────┐      ┌─────────┐ ┌────┐       ┌─────────┬───┬──────┐
│ Rock 259 │  →   │ Rock    │ │ ... │   →   │ Rock    │ 11│ 282  │
│ Jazz 545 │      │ Rock    │ │ ... │       │ Jazz    │  3│ 489  │
│ Rock 182 │      │ Jazz    │ │ ... │       │ Electr. │  6│ 316  │
│ Electr...│      │ Electr. │ │ ... │       └─────────┴───┴──────┘
│ ...      │      └─────────┘ └────┘
```


In [ ]:
import duckdb
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

# 重建数据集 / Rebuild dataset
conn = duckdb.connect()
conn.sql("""
CREATE TABLE artist (artist_id INT PRIMARY KEY, name VARCHAR, country VARCHAR);
INSERT INTO artist VALUES
    (1,'The Beatles','UK'), (2,'Pink Floyd','UK'), (3,'Miles Davis','US'),
    (4,'Daft Punk','FR'), (5,'Radiohead','UK'), (6,'Anonymous Artist',NULL);

CREATE TABLE album (album_id INT PRIMARY KEY, title VARCHAR, artist_id INT, year INT);
INSERT INTO album VALUES
    (1,'Abbey Road',1,1969), (2,'The Dark Side of the Moon',2,1973),
    (3,'The Wall',2,1979), (4,'Kind of Blue',3,1959),
    (5,'Discovery',4,2001), (6,'Random Access Memories',4,2013),
    (7,'OK Computer',5,1997), (8,'Demos (unreleased)',6,2024);

CREATE TABLE track (
    track_id INT PRIMARY KEY, name VARCHAR, album_id INT,
    genre VARCHAR, seconds INT, price DECIMAL(4,2)
);
INSERT INTO track VALUES
    (1,'Come Together',1,'Rock',259,0.99), (2,'Something',1,'Rock',182,0.99),
    (3,'Here Comes the Sun',1,'Rock',185,0.99), (4,'Time',2,'Rock',413,1.29),
    (5,'Money',2,'Rock',382,1.29), (6,'Us and Them',2,'Rock',460,1.29),
    (7,'Another Brick in the Wall',3,'Rock',239,1.29),
    (8,'Comfortably Numb',3,'Rock',382,1.29),
    (9,'So What',4,'Jazz',545,1.49), (10,'Freddie Freeloader',4,'Jazz',586,1.49),
    (11,'Blue in Green',4,'Jazz',337,1.49),
    (12,'One More Time',5,'Electronic',320,1.29),
    (13,'Aerodynamic',5,'Electronic',213,1.29),
    (14,'Digital Love',5,'Electronic',301,1.29),
    (15,'Get Lucky',6,'Electronic',369,1.29),
    (16,'Instant Crush',6,'Electronic',337,1.29),
    (17,'Lose Yourself to Dance',6,'Electronic',353,1.29),
    (18,'Paranoid Android',7,'Rock',384,1.29),
    (19,'Karma Police',7,'Rock',261,1.29),
    (20,'No Surprises',7,'Rock',228,1.29),
    (21,'Untitled Demo 1',8,'Rock',180,0.50),
    (22,'Untitled Demo 2',8,'Rock',195,0.50);

CREATE TABLE customer (customer_id INT PRIMARY KEY, name VARCHAR, country VARCHAR, email VARCHAR);
INSERT INTO customer VALUES
    (1,'Alice Chen','US','alice@example.com'), (2,'Bob Smith','UK','BOB@example.com'),
    (3,'Charlie Davis','US','charlie@example.com'),
    (4,'Diana Park','DE','diana@example.com'), (5,'Ethan Miller','US','ethan@example.com'),
    (6,'Fiona Wong','JP',NULL);

CREATE TABLE invoice (
    invoice_id INT PRIMARY KEY, customer_id INT, track_id INT,
    invoice_date DATE, quantity INT
);
INSERT INTO invoice VALUES
    (1,1,1,DATE '2026-01-05',1), (2,1,9,DATE '2026-01-05',2),
    (3,2,4,DATE '2026-01-10',1), (4,2,18,DATE '2026-01-10',1),
    (5,3,5,DATE '2026-02-12',1), (6,3,6,DATE '2026-02-12',1),
    (7,3,7,DATE '2026-02-12',3), (8,4,15,DATE '2026-02-20',1),
    (9,4,12,DATE '2026-02-20',1), (10,4,13,DATE '2026-02-20',1),
    (11,5,9,DATE '2026-03-01',1), (12,5,10,DATE '2026-03-01',1),
    (13,5,11,DATE '2026-03-01',1), (14,5,4,DATE '2026-03-05',2),
    (15,1,18,DATE '2026-03-15',1), (16,1,19,DATE '2026-03-15',1),
    (17,2,15,DATE '2026-04-01',1), (18,3,14,DATE '2026-04-10',2),
    (19,4,8,DATE '2026-05-02',1), (20,5,1,DATE '2026-05-20',1);
""")

print(f"duckdb : {duckdb.__version__}")
print("tables :", conn.sql("SHOW TABLES").df()["name"].tolist())


In [ ]:
# 最简单的 GROUP BY：按类型统计歌曲数 / Tracks per genre
conn.sql("""
    SELECT
        genre,
        COUNT(*)                 AS n_tracks,
        ROUND(AVG(seconds), 1)   AS avg_seconds,
        ROUND(AVG(price), 3)     AS avg_price
    FROM track
    GROUP BY genre
    ORDER BY n_tracks DESC;
""").df()


每个 `genre` 一行，聚合自动算出每组的 `n_tracks` / `avg_seconds` / `avg_price`。
One row per `genre`; aggregates summarize each group.

### 一句话规则 / The rule

> `GROUP BY` 之后，**`SELECT` 里只能出现两种东西**：
> 1. `GROUP BY` 列表里的列
> 2. 聚合函数（`COUNT`, `SUM`, ...）

> After `GROUP BY`, only two things may appear in `SELECT`: grouping columns, and aggregates.

打破这条规则 → 报错（或者更糟，得到"任意一行"的乱数据）。
Breaking this gives an error (or worse, silent wrong data).


In [ ]:
# 演示错误：试图 SELECT 一个非聚合非分组列 / Demo: violates the rule
# This errors in DuckDB / Postgres
try:
    conn.sql("""
        SELECT
            genre,
            name,        -- ❌ 既不在 GROUP BY 也不是聚合
            COUNT(*)
        FROM track
        GROUP BY genre;
    """).df()
except Exception as e:
    print("Error:", str(e).split("\n")[0])


> ⚠ **MySQL 的"邪恶默认"** / MySQL's evil default
> MySQL 5.6 之前的默认 `ONLY_FULL_GROUP_BY=OFF` 允许上面那种写法——**不报错，返回任意一行的 name**。这是数据 bug 的常见源头。MySQL 8+ 默认开启严格模式。
> Pre-5.6 MySQL silently returned an arbitrary row's `name` — a famous bug source. MySQL 8+ now defaults to strict.


<a id="2"></a>
## 2. `WHERE` vs `HAVING` ⭐ —— 面试必考

| 子句 / Clause | 在 GROUP BY 之前/之后 | 能用聚合吗 / Aggregates? |
|---|---|---|
| `WHERE`  | **之前** → 过滤行 | ❌ 不能 |
| `HAVING` | **之后** → 过滤组 | ✅ 可以 |

```sql
SELECT genre, COUNT(*)
FROM   track
WHERE  price >= 1.00      ← 过滤行（每首歌单独看）/ filter rows
GROUP BY genre
HAVING COUNT(*) > 3       ← 过滤组（按组聚合后才知道）/ filter groups
ORDER BY 2 DESC;
```


In [ ]:
# 这俩组合：先 WHERE 筛行，再 GROUP，再 HAVING 筛组
# Pipeline: WHERE → GROUP → HAVING
conn.sql("""
    SELECT
        genre,
        COUNT(*) AS n_tracks,
        ROUND(AVG(price), 3) AS avg_price
    FROM track
    WHERE price >= 1.00                -- 排除便宜的 demo / drop cheap demos
    GROUP BY genre
    HAVING COUNT(*) > 3                -- 只看有 > 3 首歌的 genre / only genres with >3 tracks
    ORDER BY n_tracks DESC;
""").df()


**SQL 子句的逻辑执行顺序**（不是书写顺序！）：
**Logical execution order** (different from written order!):

```
FROM  →  WHERE  →  GROUP BY  →  HAVING  →  SELECT  →  DISTINCT  →  ORDER BY  →  LIMIT
```

这就是为什么：
This is why:

- `WHERE` **不能用** `GROUP BY` 之后才有的聚合（执行时聚合还没算出来）
- `HAVING` 可以用聚合
- `ORDER BY` 可以用 `SELECT` 里的别名（它在 SELECT 之后跑）

> 💡 **面试一句话答 / One-line answer**
> "WHERE filters rows before grouping; HAVING filters groups after aggregation."


<a id="3"></a>
## 3. `SELECT` 里能放什么 —— "对齐规则"

错误示例：
Wrong example (already showed above) — but let me reinforce it:

```sql
-- ❌ 错的：name 不在 GROUP BY 也不是聚合
SELECT genre, name, COUNT(*) FROM track GROUP BY genre;

-- ✅ 对的 (1)：把 name 加进 GROUP BY → 但意义改变（按 genre × name 分组）
SELECT genre, name, COUNT(*) FROM track GROUP BY genre, name;

-- ✅ 对的 (2)：用聚合把 name "塌缩"
SELECT genre,
       STRING_AGG(name, ', ') AS sample_names,
       COUNT(*)
FROM track
GROUP BY genre;
```

`STRING_AGG`（DuckDB/Postgres）/ `GROUP_CONCAT`（MySQL）/ `LISTAGG`（Oracle/Snowflake）都是同一回事——把组内字符串拼成一个。
`STRING_AGG` / `GROUP_CONCAT` / `LISTAGG` are different dialect names for the same op.


In [ ]:
# STRING_AGG: 每个 genre 列出全部歌名 / List all track names per genre
conn.sql("""
    SELECT
        genre,
        COUNT(*) AS n_tracks,
        STRING_AGG(name, ', ' ORDER BY seconds) AS tracks
    FROM track
    GROUP BY genre
    ORDER BY n_tracks DESC;
""").df()


**字符串聚合在 DS 里超有用**：拼接每个用户的所有 event_type、每个订单的所有 SKU、每个 group 的 ID 集合。
String aggregation is incredibly handy in DS — concat all event types per user, all SKUs per order, etc.


<a id="4"></a>
## 4. `SELECT DISTINCT` vs `GROUP BY`

**面试常考题**。在"只想去重"的场景下，**两者结果完全一样**：
Common interview Q. When you just want dedup, they're identical:

```sql
SELECT DISTINCT genre FROM track;
SELECT genre FROM track GROUP BY genre;
```

但 `GROUP BY` **能继续聚合**，`DISTINCT` 不能：
But `GROUP BY` lets you aggregate further; `DISTINCT` can't:

```sql
-- ✅ GROUP BY 可以
SELECT genre, COUNT(*) FROM track GROUP BY genre;

-- ❌ DISTINCT 做不到
```

| 场景 / Use case | 选谁 / Pick |
|---|---|
| 仅去重 / Just dedup | `DISTINCT`（更易读）|
| 去重 + 聚合 / Dedup + aggregate | `GROUP BY` |
| 性能 / Performance | 大多数现代 DB 优化器**视它们等价**——挑可读的 |

**性能上**：`DISTINCT` 和 `GROUP BY (无聚合)` 在 PostgreSQL / DuckDB / BigQuery 里**经过优化是同一个执行计划**。
Optimizers compile both to the same plan on modern DBs.


In [ ]:
# 两个查询结果完全一致 / Identical results
a = conn.sql("SELECT DISTINCT genre FROM track ORDER BY genre").df()
b = conn.sql("SELECT genre FROM track GROUP BY genre ORDER BY genre").df()
print("--- DISTINCT ---")
print(a)
print("\n--- GROUP BY ---")
print(b)
print("\nidentical?", a.equals(b))


<a id="5"></a>
## 5. 多列分组 / Multi-column GROUP BY


In [ ]:
# 按 (artist, genre) 分组（先要 JOIN 上 artist 名）
# Group by (artist, genre) — needs JOIN
conn.sql("""
    SELECT
        ar.name              AS artist,
        t.genre,
        COUNT(*)             AS n_tracks,
        ROUND(AVG(t.seconds), 0) AS avg_sec
    FROM track AS t
    JOIN album  AS a  ON t.album_id  = a.album_id
    JOIN artist AS ar ON a.artist_id = ar.artist_id
    GROUP BY ar.name, t.genre
    ORDER BY ar.name, n_tracks DESC;
""").df()


**多列 GROUP BY** 给你"分类透视"——每一组 = 一个 `(artist, genre)` 组合。
Multi-column GROUP BY gives you a pivot — one row per `(artist, genre)` combination.

> **DuckDB / Postgres 16+ 简化**：`GROUP BY ALL` 自动用 `SELECT` 里所有非聚合列分组：
> Modern shortcut: `GROUP BY ALL` auto-groups by all non-aggregate columns.
> ```sql
> SELECT ar.name, t.genre, COUNT(*)
> FROM   ... JOIN ...
> GROUP BY ALL;   -- 自动 = GROUP BY ar.name, t.genre
> ```


<a id="6"></a>
## 6. 条件聚合 ⭐ / Conditional Aggregation

**面试 ★★★★★** —— "用 SQL 算转化漏斗"、"分组里某条件的占比"、"A/B 测试每组的成功率"——全是条件聚合。
**Top-5 interview pattern.** Funnels, conditional rates, A/B group stats — all conditional aggregation.

### 6.1 两种等价写法 / Two equivalent forms

```sql
-- (A) 经典写法：SUM(CASE WHEN ...)  / Classic pattern
SUM(CASE WHEN price >= 1.29 THEN 1 ELSE 0 END) AS expensive_count

-- (B) 现代写法：FILTER (WHERE ...)  / Modern (Postgres, DuckDB, SQLite 3.30+)
COUNT(*) FILTER (WHERE price >= 1.29)          AS expensive_count
```

两者语义相同，**`FILTER` 更简洁**，但 MySQL 不支持，所以**面试两种都要会**。
Same semantics; `FILTER` is cleaner but MySQL doesn't support it. Know both.


In [ ]:
# 同一组里多个"条件聚合" / Multiple conditional aggregates in one group
conn.sql("""
    SELECT
        genre,
        COUNT(*)                                                AS total,
        SUM(CASE WHEN price >= 1.29 THEN 1 ELSE 0 END)          AS expensive_count,
        COUNT(*) FILTER (WHERE price < 1.00)                    AS cheap_count,
        AVG(CASE WHEN price >= 1.29 THEN seconds END)           AS avg_sec_when_expensive,
        ROUND(
            100.0 *
            COUNT(*) FILTER (WHERE price >= 1.29) / COUNT(*),
            1
        ) AS pct_expensive
    FROM track
    GROUP BY genre
    ORDER BY pct_expensive DESC;
""").df()


**最强大的应用：A/B 测试结果表**：
The killer use case — A/B test results table:

```sql
SELECT
    variant,
    COUNT(*)                                  AS n_users,
    COUNT(*) FILTER (WHERE converted)         AS n_converted,
    100.0 * COUNT(*) FILTER (WHERE converted) / COUNT(*) AS conversion_rate
FROM experiment_log
GROUP BY variant;
```

一句 SQL 出"每组转化率"。
One query → per-group conversion rate.

### 6.2 转化漏斗 / Conversion funnel example

```sql
SELECT
    SUM(CASE WHEN event = 'view'     THEN 1 ELSE 0 END) AS viewed,
    SUM(CASE WHEN event = 'add_cart' THEN 1 ELSE 0 END) AS added,
    SUM(CASE WHEN event = 'checkout' THEN 1 ELSE 0 END) AS bought,
    ROUND(SUM(CASE WHEN event='add_cart' THEN 1 ELSE 0 END) * 100.0
        / NULLIF(SUM(CASE WHEN event='view' THEN 1 ELSE 0 END), 0), 1)
        AS view_to_cart_pct
FROM events;
```

`NULLIF(x, 0)` 是 **防 0 除**的标准技巧：当分母为 0 时返回 NULL（NULL 除任何 = NULL → 看到 NULL 立刻知道分母为 0）。
`NULLIF(x, 0)` is the standard divide-by-zero guard — turns 0 into NULL so the division becomes NULL (visible signal).


<a id="7"></a>
## 7. 进阶聚合函数 / Advanced Aggregates

除了 `COUNT/SUM/AVG/MIN/MAX`，DS 里这几个**特别有用**：
Beyond the basic five, these are DS gold:

| 函数 / Function | 作用 | 用例 |
|---|---|---|
| `STDDEV(col)`, `VAR(col)` | 标准差 / 方差 | 找异常波动 |
| `MEDIAN(col)` | 中位数 | 对异常值稳健 |
| `QUANTILE_CONT(col, p)` / `PERCENTILE_CONT(p)` | 分位数 | 算 P50/P95/P99 |
| `MODE() WITHIN GROUP (ORDER BY col)` | 众数 | 找最常见值 |
| `STRING_AGG(col, sep)` | 字符串拼接 | 列出组内所有值 |
| `ARRAY_AGG(col)` | 数组聚合 | 收集组内所有值（保留顺序）|
| `BOOL_OR(col)`, `BOOL_AND(col)` | 任意/全部为真 | "用户**任一次**点击过 X" |
| `COUNT(DISTINCT col)` | 去重计数 | UV、unique users |

下面我们 hit 几个最实用的：
Let's hit the most useful:


In [ ]:
# STDDEV / VAR / 分位数 / Mode
conn.sql("""
    SELECT
        genre,
        ROUND(AVG(seconds), 1)                          AS mean,
        ROUND(STDDEV(seconds), 1)                       AS stddev,
        MIN(seconds)                                    AS min,
        QUANTILE_CONT(seconds, 0.25)                    AS p25,
        QUANTILE_CONT(seconds, 0.50)                    AS p50_median,
        QUANTILE_CONT(seconds, 0.75)                    AS p75,
        QUANTILE_CONT(seconds, 0.95)                    AS p95,
        MAX(seconds)                                    AS max
    FROM track
    GROUP BY genre
    ORDER BY mean DESC;
""").df()


In [ ]:
# ARRAY_AGG: 收集每个艺术家的所有专辑年份 / Years of each artist's albums
conn.sql("""
    SELECT
        ar.name AS artist,
        ARRAY_AGG(a.year ORDER BY a.year) AS years
    FROM album  AS a
    JOIN artist AS ar USING (artist_id)
    GROUP BY ar.artist_id, ar.name
    ORDER BY ar.name;
""").df()


In [ ]:
# BOOL_OR / BOOL_AND: 任一/全部为真 / Any / all true
# "每个 genre 是否至少有一首歌 ≥ 5 分钟"
conn.sql("""
    SELECT
        genre,
        COUNT(*)                                     AS n_tracks,
        BOOL_OR(seconds >= 300)                      AS has_long_track,
        BOOL_AND(price >= 1.00)                       AS all_paid
    FROM track
    GROUP BY genre;
""").df()


<a id="8"></a>
## 8. `ROLLUP` —— 自动小计 + 总计 / Subtotals + Grand Total

业务报表常见需求："每个 (country, genre) 一行，**然后每个 country 一个小计**，最后一个**总计行**"。
Classic BI need: per (country, genre), plus per-country subtotal, plus a grand total at the bottom.

手写要 `UNION` 三次。**`ROLLUP` 一行搞定**：
Writing this manually needs 3-way UNION. `ROLLUP` does it in one line.

```sql
GROUP BY ROLLUP(country, genre)
```

`ROLLUP(a, b, c)` 等价于：
```
(a, b, c)
(a, b)        ← b "summed up", c "rolled away"
(a)
()            ← grand total
```


In [ ]:
# 每个 (artist country × track genre) 的销售额 + 小计 + 总计
# Sales per (country, genre) + subtotals + grand total
conn.sql("""
    SELECT
        ar.country,
        t.genre,
        COUNT(*)                                            AS n_tracks_sold,
        ROUND(SUM(i.quantity * t.price), 2)                 AS revenue
    FROM invoice AS i
    JOIN track   AS t  ON i.track_id  = t.track_id
    JOIN album   AS a  ON t.album_id  = a.album_id
    JOIN artist  AS ar ON a.artist_id = ar.artist_id
    GROUP BY ROLLUP(ar.country, t.genre)
    ORDER BY ar.country NULLS LAST, t.genre NULLS LAST;
""").df()


**读输出**：
- 每个 `(country, genre)` 一行
- 每个 country 的 `genre = NULL` 行 = **该国小计**
- 最后 `country = NULL, genre = NULL` 行 = **总计**

How to read: NULL in a grouping column = "this row is a subtotal/total over that dim".

### `GROUPING()` 函数：分清 NULL 是"原数据"还是"小计标记"

如果原始数据里 `genre` 本身有 NULL，怎么判断这行 NULL 是"原本就空"还是"小计"？用 `GROUPING(col)`：
If the source data has real NULLs, how to tell them from rollup NULLs? Use `GROUPING(col)`:

- `GROUPING(col) = 0` → 原始值（含 NULL）
- `GROUPING(col) = 1` → 这一行是该列的"小计/总计"


In [ ]:
conn.sql("""
    SELECT
        COALESCE(ar.country, '—') AS country,
        COALESCE(t.genre,    '—') AS genre,
        GROUPING(ar.country)      AS rolling_country,
        GROUPING(t.genre)         AS rolling_genre,
        COUNT(*)                  AS n,
        CASE
            WHEN GROUPING(ar.country) = 1 AND GROUPING(t.genre) = 1 THEN 'GRAND TOTAL'
            WHEN GROUPING(t.genre)    = 1                            THEN 'subtotal per country'
            ELSE 'leaf'
        END                       AS row_kind
    FROM invoice AS i
    JOIN track   AS t  ON i.track_id  = t.track_id
    JOIN album   AS a  ON t.album_id  = a.album_id
    JOIN artist  AS ar ON a.artist_id = ar.artist_id
    GROUP BY ROLLUP(ar.country, t.genre)
    ORDER BY ar.country NULLS LAST, t.genre NULLS LAST;
""").df()


<a id="9"></a>
## 9. `CUBE` —— 所有维度组合 / All-dimension Crosstab

`CUBE(a, b)` 比 `ROLLUP(a, b)` 多一种组合：单独按 `b` 聚合。
`CUBE(a, b)` adds one more combo vs `ROLLUP`: aggregating by `b` alone.

```
CUBE(a, b)  →  (a, b)
                (a)
                (b)       ← extra
                ()

ROLLUP(a, b) →  (a, b)
                (a)
                ()
```

业务上："**同时**给我按 country 的小计 **+** 按 genre 的小计 **+** 总计"——CUBE 一次搞定。
Use case: "give me all the subtotals at once" — by country alone AND by genre alone AND grand total.


In [ ]:
conn.sql("""
    SELECT
        COALESCE(ar.country, 'ALL') AS country,
        COALESCE(t.genre,    'ALL') AS genre,
        COUNT(*) AS n
    FROM invoice AS i
    JOIN track   AS t  ON i.track_id  = t.track_id
    JOIN album   AS a  ON t.album_id  = a.album_id
    JOIN artist  AS ar ON a.artist_id = ar.artist_id
    GROUP BY CUBE(ar.country, t.genre)
    ORDER BY country, genre;
""").df()


现在能看到：
- 每个 `(country, genre)` 一行
- 每个 country 的 "ALL" genre 行（country 维度的小计）
- 每个 genre 的 "ALL" country 行（**ROLLUP 没有的**）
- 总计 "ALL", "ALL"

Now you see country subtotals, genre subtotals (NEW vs ROLLUP), and grand total.


<a id="10"></a>
## 10. `GROUPING SETS` —— 自由组合

`GROUPING SETS((a, b), (a), (b))` 让你**精确指定**要哪些维度组合。
`GROUPING SETS` lets you pick exact combos.

`ROLLUP(a, b)` = `GROUPING SETS((a, b), (a), ())`
`CUBE(a, b)`   = `GROUPING SETS((a, b), (a), (b), ())`


In [ ]:
# 只要 (country, genre) 和总计，不要中间的小计
# Only (country, genre) and grand total — no subtotals
conn.sql("""
    SELECT
        COALESCE(ar.country, 'TOTAL') AS country,
        COALESCE(t.genre,    'TOTAL') AS genre,
        COUNT(*) AS n
    FROM invoice AS i
    JOIN track   AS t  ON i.track_id  = t.track_id
    JOIN album   AS a  ON t.album_id  = a.album_id
    JOIN artist  AS ar ON a.artist_id = ar.artist_id
    GROUP BY GROUPING SETS ((ar.country, t.genre), ())
    ORDER BY country, genre;
""").df()


<a id="11"></a>
## 11. ⚠ NULL 在 GROUP BY 中的行为

**GROUP BY 把所有 NULL 当成"一组"**——和 1.1 节学的 "NULL 不等于任何值（包括 NULL）" 看起来矛盾！
**GROUP BY treats all NULLs as one group** — apparently contradicting "NULL != NULL" from 1.1.

但这是规则——SQL 标准明确规定：
But this is the rule — explicit in the SQL standard:

> 在 `GROUP BY`、`DISTINCT`、`UNION`、`INTERSECT`、`EXCEPT` 中，所有 NULL **相等**。
> In `GROUP BY`, `DISTINCT`, `UNION`, etc., all NULLs are **equal**.

在比较运算（`=`、`<`、`<>`）里它们仍然 NULL ≠ NULL。**记牢这个分裂的语义**。
In equality/comparison they still don't match. Memorize this split semantics.


In [ ]:
# 演示：按 country 分组，NULL 自成一组
# All NULLs collapse into one group
conn.sql("""
    SELECT country, COUNT(*) AS n
    FROM customer
    GROUP BY country
    ORDER BY country NULLS LAST;
""").df()


6 个客户里 Fiona Wong 的 `country = 'JP'`，**没有 NULL country 的客户**——所以这里没演示。但你可以想象：如果有 3 个 NULL country 的客户，GROUP BY 会显示一行 `country = NULL, n = 3`。
None of our customers have a NULL country, so the demo above doesn't show it. But if 3 did, GROUP BY would collapse them into one row.


<a id="12"></a>
## 12. 实战：销售分析报表 / Sales Analytics Report

完整给老板做一份"音乐商店 2026 上半年销售报表"。
Build a full "Music Store H1 2026 sales report".


In [ ]:
# Q1: 全局指标 / Overall KPIs
conn.sql("""
    SELECT
        COUNT(*)                                          AS invoices,
        COUNT(DISTINCT customer_id)                       AS active_customers,
        SUM(i.quantity)                                   AS units_sold,
        ROUND(SUM(i.quantity * t.price), 2)               AS revenue,
        ROUND(SUM(i.quantity * t.price) / COUNT(*), 2)    AS avg_order_value
    FROM invoice AS i
    JOIN track   AS t USING (track_id);
""").df()


In [ ]:
# Q2: 按月销售 / Monthly sales
conn.sql("""
    SELECT
        DATE_TRUNC('month', invoice_date) AS month,
        COUNT(*)                              AS orders,
        ROUND(SUM(i.quantity * t.price), 2)   AS revenue
    FROM invoice AS i
    JOIN track   AS t USING (track_id)
    GROUP BY 1
    ORDER BY 1;
""").df()


In [ ]:
# Q3: 按 genre 销售 / By genre
conn.sql("""
    SELECT
        t.genre,
        COUNT(*)                                                 AS orders,
        ROUND(SUM(i.quantity * t.price), 2)                      AS revenue,
        ROUND(100.0 * SUM(i.quantity * t.price)
            / SUM(SUM(i.quantity * t.price)) OVER (), 1)         AS pct_of_total  -- 注意：用了窗口函数（1.5 节深入）
    FROM invoice AS i
    JOIN track   AS t USING (track_id)
    GROUP BY t.genre
    ORDER BY revenue DESC;
""").df()


In [ ]:
# Q4: 高客单价订单（HAVING）/ Big-ticket orders
conn.sql("""
    SELECT
        i.invoice_id,
        c.name                                  AS customer,
        ROUND(SUM(i.quantity * t.price), 2)     AS order_total,
        COUNT(*)                                AS line_items
    FROM invoice   AS i
    JOIN customer  AS c USING (customer_id)
    JOIN track     AS t USING (track_id)
    GROUP BY i.invoice_id, c.name
    HAVING SUM(i.quantity * t.price) > 3.0
    ORDER BY order_total DESC;
""").df()


In [ ]:
# Q5: 国家×Genre 交叉表 + 小计 + 总计 (ROLLUP)
conn.sql("""
    SELECT
        COALESCE(ar.country, 'ALL_COUNTRIES') AS country,
        COALESCE(t.genre,    'ALL_GENRES')    AS genre,
        COUNT(*)                              AS units,
        ROUND(SUM(i.quantity * t.price), 2)   AS revenue
    FROM invoice AS i
    JOIN track   AS t  ON i.track_id  = t.track_id
    JOIN album   AS a  ON t.album_id  = a.album_id
    JOIN artist  AS ar ON a.artist_id = ar.artist_id
    GROUP BY ROLLUP(ar.country, t.genre)
    ORDER BY country, genre;
""").df()


In [ ]:
# Q6: 用户类型条件聚合 / Customer-level conditional stats
# "每个客户在 Q1 (Jan–Mar) 和 Q2 (Apr–Jun) 各花了多少"
conn.sql("""
    SELECT
        c.name,
        ROUND(SUM(CASE WHEN invoice_date < DATE '2026-04-01'
                   THEN i.quantity * t.price ELSE 0 END), 2)    AS q1_revenue,
        ROUND(SUM(CASE WHEN invoice_date >= DATE '2026-04-01'
                   THEN i.quantity * t.price ELSE 0 END), 2)    AS q2_revenue,
        ROUND(SUM(i.quantity * t.price), 2)                     AS total_revenue
    FROM invoice  AS i
    JOIN customer AS c USING (customer_id)
    JOIN track    AS t USING (track_id)
    GROUP BY c.customer_id, c.name
    ORDER BY total_revenue DESC;
""").df()


In [ ]:
# Q7: 每种 genre 的"价格分位数" / Price percentiles per genre
conn.sql("""
    SELECT
        genre,
        QUANTILE_CONT(price, 0.25) AS p25,
        QUANTILE_CONT(price, 0.50) AS median,
        QUANTILE_CONT(price, 0.75) AS p75,
        MIN(price)                  AS cheapest,
        MAX(price)                  AS priciest
    FROM track
    GROUP BY genre
    ORDER BY median DESC;
""").df()


In [ ]:
# Q8: 每个客户的"购买物清单" / All tracks each customer bought
conn.sql("""
    SELECT
        c.name,
        COUNT(*)                                   AS purchases,
        STRING_AGG(DISTINCT t.name, ', '
                   ORDER BY t.name)                AS unique_tracks
    FROM invoice  AS i
    JOIN customer AS c USING (customer_id)
    JOIN track    AS t USING (track_id)
    GROUP BY c.customer_id, c.name
    ORDER BY purchases DESC;
""").df()


<a id="13"></a>
## 13. 小结 / Summary

### 概念地图 / Concept map

```
GROUP BY
  │
  ├── 单列 / 多列分组
  ├── SELECT 对齐规则：只能 (分组列 ∪ 聚合)
  ├── HAVING：在聚合后过滤组（WHERE 不行）
  │
  ├── 条件聚合 ⭐
  │     ├── SUM(CASE WHEN ...) ELSE 0 END)
  │     └── COUNT(*) FILTER (WHERE ...)  ← 现代写法
  │
  ├── 进阶聚合
  │     ├── STDDEV / VAR
  │     ├── QUANTILE_CONT  (P50 / P95 / P99)
  │     ├── STRING_AGG / ARRAY_AGG
  │     ├── BOOL_OR / BOOL_AND
  │     └── COUNT(DISTINCT)
  │
  └── 多维 / Multi-dim
        ├── ROLLUP(a, b)       小计 + 总计
        ├── CUBE(a, b)         全部组合
        └── GROUPING SETS(...) 任意子集
              + GROUPING() 标记是否"小计行"
```

### 💡 必背 / Must-remember

| Pattern | Why |
|---|---|
| `WHERE` 在分组前；`HAVING` 在分组后 | 执行顺序差异 |
| `SELECT` 里非聚合列**必须** in `GROUP BY` | 不然语义不明 |
| `GROUP BY` 把所有 NULL 当一组 | 与 `=` 不同！|
| `COUNT(*)` 含 NULL；`COUNT(col)` 不含 | 1.1 节回顾 |
| `NULLIF(x, 0)` 是除 0 的标准守护 | 别忘 |
| `FILTER (WHERE ...)` ≡ `SUM(CASE WHEN ...)` | 后者通用 |

### 💡 工业速查 / Industry cheat sheet

```sql
-- 转化率一行 / Conversion rate one-liner
SELECT
    100.0 * COUNT(*) FILTER (WHERE converted) / COUNT(*) AS conv_rate
FROM events;

-- 按月汇总 + ROLLUP / Monthly + grand total
SELECT
    DATE_TRUNC('month', ts) AS month,
    SUM(amount)
FROM tx
GROUP BY ROLLUP(month);

-- 每用户 P50 / P95 延迟 / Per-user latency percentiles
SELECT
    user_id,
    QUANTILE_CONT(latency_ms, 0.5)  AS p50,
    QUANTILE_CONT(latency_ms, 0.95) AS p95
FROM api_log
GROUP BY user_id;

-- 把每用户所有 event_type 拼成数组 / Collect events per user
SELECT
    user_id,
    ARRAY_AGG(event_type ORDER BY ts) AS event_sequence
FROM events
GROUP BY user_id;
```

### 💡 面试必答 / Interview must-knows

1. **WHERE 在 GROUP 前，HAVING 在 GROUP 后** —— 一句话答完
2. **`SELECT` 非聚合列必须 `GROUP BY`** —— 否则 MySQL 老版本会"静默返回任意一行"
3. **`COUNT(DISTINCT x)` ≠ `COUNT(x)`**
4. **条件聚合的两种写法** —— CASE WHEN 通用；FILTER 简洁
5. **`NULLIF(分母, 0)`** 防 0 除
6. **`ROLLUP`/`CUBE`/`GROUPING SETS`** 一次出多维报表，避免 UNION 拼

### 下一节预告 / Next up

**Part 1.4 · 子查询 & CTE** —— Subquery、Correlated subquery、`WITH RECURSIVE`、CTE 把复杂 SQL 写得像 Python 函数一样可读。
**Part 1.4 · Subquery & CTE** — make complex SQL as readable as Python functions.
